<a href="https://colab.research.google.com/github/brynndafoe02/dsci552_summer2026/blob/master/Dafoe_Brynn_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

DSCI 552 Final Project

Name: Brynn Dafoe GitHub Username: brynndafoe02 USD ID: 3109-6692-10

In [23]:
import cv2
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as IPipeline # for SMOTE / 1.b.iv
from imblearn.under_sampling import RandomUnderSampler
from itertools import combinations
import math
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
from pathlib import Path
from PIL import Image
import re
import seaborn as sns

from sklearn import datasets
from sklearn.cluster import KMeans
from sklearn.datasets import make_classification, make_regression
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression, LogisticRegressionCV, RidgeCV, LassoCV
from sklearn.metrics import silhouette_score, hamming_loss, RocCurveDisplay, roc_auc_score, accuracy_score, classification_report, confusion_matrix, mean_squared_error, r2_score, mean_absolute_error, precision_score, recall_score, f1_score
from sklearn.model_selection import GridSearchCV, train_test_split, StratifiedKFold, cross_val_score, RepeatedKFold, KFold
from sklearn.multiclass import OneVsRestClassifier
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline as SPipeline # for StandardScaler / 1.b.iii
from sklearn.preprocessing import StandardScaler, label_binarize, OneHotEncoder
from sklearn.svm import SVC, LinearSVC
from sklearn.tree import DecisionTreeClassifier, plot_tree, _tree

#from skmultilearn.problem_transform import LabelPowerset
import statsmodels.api as sm
import statsmodels.formula.api as smf

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.applications import ResNet50, ResNet101, EfficientNetB0, VGG16
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
from tensorflow.keras.applications.efficientnet import preprocess_input as effnet_preprocess
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg_preprocess

import xgboost as xgb

In [24]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [25]:
data_path = "/content/drive/MyDrive/defungi"

Objective:
- Trying to build a classifier that distinguishes images of FIVE types of waste

Data Exploration and Pre-Processing
- Images are numbered in each folder
- Select FIRST 80% of the images in EACH FOLDER as your TRAINING set
- - Use the rest as the TEST set
  - Can you encode classes using one-hot encoding
- In case all the images do not have the same size, ZERO-PAD or RESIZE the images in your dataset
- - This can be done using various tools, including OpenCV

In [26]:
IMAGE_WIDTH = 224
IMAGE_HEIGHT = 224

In [27]:
# 80% Training, 20% Testing
    # Will take the 20% Validation Set from the training set now as well
    # So for the 80% training: 64% for actual training, 16% for VS

In [28]:
defungi_path = Path(data_path)
waste_class_names = ["C1", "C2", "C3", "C4", "C5"]

In [29]:
# to make sure the images get sorted correctly
    # aka: making sure H1_100a_3.jpg comes before H1_101a_1.jpg, etc.
def natural_sorting(path_to_image):
    return [
        int(c) if c.isdigit() else c.lower()
        for c in re.split(r'(\d+)', path_to_image.name)
    ]

In [30]:
training_paths = []
training_labels = []

testing_paths = []
testing_labels = []

valset_paths = []
valset_labels = []

# parallel lists: ex -> training_paths has the image paths, training_labels has the class labels
# I'm keeping these separate cause I don't want to deal with a dictionary


In [31]:
cw_index = 0
for waste_class in waste_class_names:
    waste_class_folder = defungi_path / waste_class
    # makes the path to the inner folder from the outer defungi folder
        # so now "../data/defungi/C1" for example

    image_files_list = list(waste_class_folder.glob("*.jpg"))
    sorted_filenames = sorted(image_files_list, key=natural_sorting) # NATURALLY sort all the image files

    tt_split_index = int(0.80 * len(sorted_filenames))

    # training + validation all together first
    training_validation_set = sorted_filenames[:tt_split_index]
    # testing set, can leave this alone
    testing_image_paths = sorted_filenames[tt_split_index:]

    # splitting into true training set and true validation set
    # random 20% of train+val set will go to validation_image_paths, other 80% to training
    training_image_paths, validation_image_paths = train_test_split(training_validation_set, test_size=0.20, random_state=42, shuffle=True)

    #####

    training_paths.extend(training_image_paths)
    train_class_label_list = [cw_index] * len(training_image_paths)
    training_labels.extend(train_class_label_list)

    testing_paths.extend(testing_image_paths)
    test_class_label_list = [cw_index] * len(testing_image_paths)
    testing_labels.extend(test_class_label_list)

    valset_paths.extend(validation_image_paths)
    val_class_label_list = [cw_index] * len(validation_image_paths)
    valset_labels.extend(val_class_label_list)

    cw_index+=1


In [32]:
# resnet50, resnet101, efficientnetb0, and vgg16 take: 224 x 224 picels with 3 color channels
    # resnet + efficientnet -> expect RGB
    # vgg16 -> expects BGR
        # preprocess will convert back later, to make simple going to convert all to RGB now
def resize_zeropad_images(image_paths):
    fixed_images = []
    for image_path in image_paths:
        image = cv2.imread(str(image_path))
        if image is None:
            raise ValueError(f"Could not load image")

        # convert all to RGB
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        og_h, og_w = image.shape[:2]

        scale = min(IMAGE_HEIGHT / og_h, IMAGE_WIDTH / og_w)

        # new dimensions
        new_w = int(og_w * scale)
        new_h = int(og_h * scale)

        # resize image
        resized_image = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_AREA)

        top = (IMAGE_HEIGHT - new_h) // 2
        bottom = IMAGE_HEIGHT - new_h - top
        left = (IMAGE_WIDTH - new_w) // 2
        right = IMAGE_WIDTH - new_w - left

        # zero padding image
        padded_image = cv2.copyMakeBorder(
            resized_image, top, bottom, left, right,
            borderType=cv2.BORDER_CONSTANT,
            value=[0, 0, 0]
        )

        fixed_images.append(padded_image)

    return fixed_images


In [33]:
X_train = np.array(resize_zeropad_images(training_paths))
X_test= np.array(resize_zeropad_images(testing_paths))
X_val = np.array(resize_zeropad_images(valset_paths))

encoder = OneHotEncoder(sparse_output=False)

Y_train = encoder.fit_transform(np.array(training_labels).reshape(-1, 1))
Y_test = encoder.fit_transform(np.array(testing_labels).reshape(-1, 1))
Y_val = encoder.fit_transform(np.array(valset_labels).reshape(-1, 1))

Transfer Learning i
- Transfer learning for small image datasets
- - uses deep learning models that are trained on large datasets as feature extractors
  - these deep networks have learned to extract meaningful features from an image using their layers, thos features can be used in learning other tasks
- Usually last / last few layers of the pre-trained network are removed, and the response of the layer before the removed layers to the images in the new dataset is used as a feature vector to train one more multiple replacement layers
- Will use pre-trained models: ResNet50, ResNet101, EfficientNetB0, and VGG16
- - will only train LAST fully connected layer, and will FREEZE all layers before them (we do not change their parameters during training)
  - will use the outputs of the penultimate layer in the original pre-trained model as the features extracted from each image

Transfer Learning ii
- To perform empirical regularization, crop, randomly zoom, rotate, flip, contrast, and translate images in your training set for image augmentation
- - can use various tools to do this (including OpenCV)

In [34]:
data_augmentation = tf.keras.Sequential([
    layers.RandomCrop(180, 180),
    layers.RandomZoom(0.2),
    layers.RandomRotation(0.2),
    layers.RandomFlip("horizontal"),
    layers.RandomContrast(0.2),
    layers.RandomTranslation(0.1, 0.1),
    layers.Resizing(224, 224)
])
# need resize at end because need 224x224

In [35]:
# ResNet50
# load base model
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
# freeze all layers in base model
base_model.trainable = False
# build final model
inputs = keras.Input(shape=(224, 224, 3))
# augment images
x = data_augmentation(inputs)
# specific preprocess for model to make images be correct for model
x = resnet_preprocess(x)
# pass augmented / pre-processed images through ResNet
x = base_model(x, training=False)
    # false keeps BatchNormalization in inference mode
x = layers.GlobalAveragePooling2D()(x)

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [36]:
# ReLu activation functions in dense layer
# L2 regularization
x = layers.Dense(256, activation="relu", kernel_regularizer=regularizers.l2(0.001))(x)
    # 256 neurons was recomended for the training size and number of classes
    # 0.001 as a middle ground penalty
# batch normalization
x = layers.BatchNormalization()(x)
# dropout rate of 20%
x = layers.Dropout(0.2)(x)
# softmax layer -> needs to be last / last layer
outputs = layers.Dense(5, activation="softmax")(x)

In [37]:
# ADAM optimizer
# multinomial cross entropy loss
resnet50_fm = keras.Model(inputs=inputs, outputs=outputs)
resnet50_fm.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
                   loss="categorical_crossentropy",
                   metrics=["accuracy"])

In [38]:
# can try any batch size, but 5 seems reasonable
history_resnet50 = resnet50_fm.fit(
    X_train,
    Y_train,
    batch_size=5,
    epochs=2,
    validation_data=(X_val, Y_val)
)

Epoch 1/2
1166/1166 ━━━━━━━━━━━━━━━━━━━━ 63s 41ms/step - accuracy: 0.6015 - loss: 1.2697 - val_accuracy: 0.7158 - val_loss: 0.9629
Epoch 2/2
1166/1166 ━━━━━━━━━━━━━━━━━━━━ 45s 38ms/step - accuracy: 0.6563 - loss: 1.0321 - val_accuracy: 0.7555 - val_loss: 0.8251
